# Partition and AutoPartition

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

## Partition

Bind a packed word to named register elements. Place registers in
declaration order, array elements in row-major order, and bits in each
word from most to least significant. The packed word is an input for
Partition and an output for its adjoint. The original shaped example
has six two-bit elements and a 37-bit word. A physical binding only
renames wire groups and emits no gates.

In [ ]:
from flagquantum.compiler.registers.layout.component_partition import (
    partition_component_program, partition_signature,
)
from flagquantum.compiler.registers.register_binding import bind_register_program

### Example Instances

In [ ]:
ports = (Register("xx", opaque_register_type(2), shape=(2, 3)),
         Register("yy", opaque_register_type(37)))
shaped = partition_signature(ports)
assert shaped[0].bitsize == 49
split = bind_register_program(partition_component_program(ports, "main", False),
                              (tuple(range(49)),))
join = bind_register_program(partition_component_program(ports, "main", True), split.outputs)
assert join.outputs == (tuple(range(49)),)
assert split.circuit.instructions == join.circuit.instructions == ()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Partition", shaped, {})])))

## AutoPartition

Adapt a child operation to packed registers, such as combining control
and data bits in one input word. Split each group, call the child and
restore the groups through inverse partitions. Mark unused slices with
`None` to retain their original wires. In left-only mode expose the
child's output ports in their declaration order and omit the inverse
partitions. That mode requires every packed slice to belong to the child.

In [ ]:
from flagquantum.compiler.registers.layout.auto_partition import AutoPartition
from flagquantum.compiler.registers.primitives.controlled_swap import controlled_swap_ir

### Example Instances

In [ ]:
packed = AutoPartition((
    ("x", (("ctrl", 1), ("x", 1))), ("y", (("y", 1),)),
))
packed_signature = Signature.build(x=2, y=1)

In [ ]:
with_unused = AutoPartition((
    ("x", (("ctrl", 1), ("x", 1), (None, 1))),
    ("y", (("y", 1),)), ("z", ((None, 2),)),
))
unused_signature = Signature.build(x=3, y=1, z=2)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("AutoPartition", packed_signature, {}),
    ("AutoPartition with unused slices", unused_signature, {}),
])))

### Call Graph

In [ ]:
root = "AutoPartition"
calls = {(root, label): 1 for label in (
    "Partition x", "Partition y", "Controlled(Swap)", "Join x", "Join y",
)}
display(SVG(render_call_graph_svg(export_call_graph_dot(root, calls))))
print("Direct calls", list(calls.values()))

### Append the bound child

Run the original packed one-bit controlled exchange on a coherent
three-qubit state. Partitioning leaves the packed wire layout intact.
The two active computational basis components exchange places.

In [ ]:
bound, unused = packed.bind({"x": (0, 1), "y": (2,)})
assert packed.restore(bound, unused) == {"x": (0, 1), "y": (2,)}
operation = FieldOperation(3, controlled_swap_ir(bound["ctrl"][0], bound["x"], bound["y"]))
values = torch.arange(1, 9, dtype=torch.float64)
initial = torch.complex(values, values.flip(0))
initial /= initial.norm()
circuit = Circuit(3, inputs=initial, dtype=initial.dtype).append(operation)
expected = initial[[0, 1, 2, 3, 4, 6, 5, 7]]
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=2e-12)